# Walking through the CONUS stream CO2 model (R to Python)
This notebook walks the *entire* CO₂-model pipeline, ported from the original R code
to Python, and runs it end-to-end on a small synthetic test basin. Each step shows the code,
explains what it does, and ties back to the exact R source lines. The flow is unchanged from R;
only implementation efficiency differs (numpy vectorization, an O(N) upstream-index map).

> Synthetic data.
> files on the UMass NAS (not in this repo). We use a 12-reach toy basin to demonstrate the flow.
> Absolute CO₂ magnitudes are illustrative; the relative patterns (e.g. river vs lake
> sensitivity) are what matter here.

## 0. Setup

Import the ported package `conus_carbon`. Run this notebook from the repo root so the package is importable.


In [ ]:
import os, sys
# make the package importable whether launched from the repo root or notebooks/
for p in ['.', '..']:
    if os.path.isdir(os.path.join(p, 'conus_carbon')):
        sys.path.insert(0, os.path.abspath(p)); break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conus_carbon import synthetic_basin, hydraulics, model, calibrate, run_model
from conus_carbon.constants import (C_GROUNDWATER_PPM, C_ATMOSPHERE_PPM,
                                    EMERGENCE_Q_M3S, PARAM_NAMES, LOWER_BOUNDS, UPPER_BOUNDS)
print('conus_carbon loaded. knobs =', PARAM_NAMES)
print('fixed constants: C_gw =', C_GROUNDWATER_PPM, 'ppm | C_atm =', C_ATMOSPHERE_PPM,
      'ppm | emergenceQ =', EMERGENCE_Q_M3S, 'm3/s')

## 1. The big picture

The pipeline has three stages:

1. `setup_hydrography` builds a per-basin table of stream *reaches* with precomputed
   physics (residence time `HRT`, gas exchange `k_co2`, hyporheic exchange `k_bz`, geometry).
2. `calibrate` / reach loop runs the forward model over the reaches (upstream -> downstream)
   for a given set of 4 knobs, then scores it: fitness = 1 / cost.
3. `run_model` applies the chosen knobs and computes the CO₂ flux to the atmosphere.

There are 4 knobs but only 2 targets (median river pCO₂, median lake pCO₂). That gap is
the *equifinality* the whole project is about.

2. `calibrate`.

## 2. Physics helpers (`hydraulics.py` from `utils.R`)

These reproduce the physics functions from `utils.R` exactly (Henry's constant, the
Ulseth-2019 gas-exchange `k_co2`, the Grant-2018/Lorke-Peeters hyporheic `k_bz`, residence time).
They are applied once when a basin is built, not inside the hot reach loop.


In [ ]:
# Henry's constant falls as water warms (less soluble)
for T in [5, 15, 25]:
    print(f'Henry @ {T:2d} C = {float(hydraulics.henry_func(T)):.4f} mol/(L*atm)')

# gas-exchange k_co2 for a steep vs gentle river reach
import numpy as np
k_steep = hydraulics.kco2_func(np.array([1.0]), np.array([0.03]), np.array([0.5]),
                               np.array([15.0]), np.array([0.0]), np.array(['River']))
k_gentle = hydraulics.kco2_func(np.array([0.2]), np.array([0.001]), np.array([1.0]),
                                np.array([15.0]), np.array([0.0]), np.array(['River']))
print(f'k_co2 steep  = {float(k_steep[0]):.2e} 1/s')
print(f'k_co2 gentle = {float(k_gentle[0]):.2e} 1/s  (steeper reach degasses faster)')

## 3. Build a test basin (`synthetic_basin.py`)

`make_synthetic_basin()` returns a 12-reach toy network with the same 24-column schema
that `setup_hydrography.R` produces (the R `select()` lists `NHDPlusID` twice, so it reads as
'25' but is 24 unique). Note the river vs lake residence time contrast, this drives
everything later.


In [ ]:
basin = synthetic_basin.make_synthetic_basin()
print('shape:', basin.shape, '| waterbody:', basin['waterbody'].value_counts().to_dict())

# residence time (HRT) is short for rivers, long for lakes
summary = basin.groupby('waterbody')['HRT'].agg(['min','median','max']).round(0)
print('\nresidence time HRT [s] by water-body:')
print(summary)

basin[['NHDPlusID','FromNode','ToNode','HydroSeq','StartFlag','waterbody','Q_cms','HRT','k_co2','k_bz']].round(4)

## 4. The physics core: `CO2_MODEL` (equation 6)

For one reach, with the knobs fixed, CO₂ is a simple mass balance (`model.R:60`):

$$C = \frac{C_{last} + \frac{Q-Q_{last}}{Q}C_{gw} + hrt\,k_{co2}C_{atm} + hrt\,k_{bz}C_{bz} + hrt\,F_{wc}}{1 + hrt\,k_{co2} + \frac{Q-Q_{last}}{Q}}$$

where `C_last` is the discharge-weighted CO₂ flowing in from upstream. Headwaters start at
groundwater `C_gw`; Great Lakes are pinned to atmospheric `C_atm`. Only `C_bz` and `F_wc` are
knobs: everything else is fixed.


In [ ]:
# one headwater reach, no upstream inflow
co2 = model.co2_model_reach(
    hrt=3500.0, Q=0.5, C_gw=C_GROUNDWATER_PPM, C_atm=C_ATMOSPHERE_PPM,
    C_bz=1000.0, k_co2=1.3e-3, k_bz=2.6e-3, F_wc=1e-4,
    divergence=0, startflag=1, waterbody='River',
    upstream_co2=np.array([]), upstream_Q=np.array([]), emergenceQ=EMERGENCE_Q_M3S)
print(f'headwater reach CO2 = {co2:.1f} ppm  (between C_atm={C_ATMOSPHERE_PPM:.0f} and C_gw={C_GROUNDWATER_PPM:.0f})')

## 5. The reach loop + fitness (`calibrate.py` from `calibrate_model.R`)

`run_forward` sorts reaches upstream -> downstream (descending `HydroSeq`) and fills each
reach's CO₂ in one pass, exactly like the R loop. Then the fitness: take the median modeled
pCO₂ over river reaches and over lake reaches, compare to the two targets, `cost = sum of
absolute errors`, `fitness = 1/cost`.

*Efficiency-only change:* R searched `which(toNode == fromNode)` inside every reach (O(N²));
we precompute an upstream-index map once (O(N)): identical results.

In [ ]:
targets = synthetic_basin.SYNTHETIC_TARGETS   # illustrative GLORICH-style targets
params = (600.0, 600.0, 1e-4, 1e-5)           # (Cbz_riv, Cbz_lake, Fwc_riv, Fwc_lake)

h = calibrate.run_forward(basin, params)      # forward pass -> CO2 per reach
ev = calibrate.evaluate(basin, params, riverCO2=targets['riverCO2'], lakeCO2=targets['lakeCO2'])
print('targets           :', targets)
print(f'median river pCO2 : {ev.river_median:8.1f} ppm')
print(f'median lake  pCO2 : {ev.lake_median:8.1f} ppm')
print(f'cost             : {ev.cost:8.1f} ppm   ->   fitness = 1/cost = {ev.fitness:.3e}')

h[['NHDPlusID','waterbody','Q_cms','HRT','CO2_ppm']].round(2)

## 6. Run the calibrated model + emissions (`run_model.py` from `run_model.R`)

`run_model` applies a chosen knob set and adds the diagnostic and emission columns, including `FCO2_gC_m2_yr`, the CO₂ flux to the atmosphere per reach.


In [ ]:
calib = dict(Cbz_riv=600.0, Cbz_lake=600.0, Fwc_riv=1e-4, Fwc_lake=1e-5)
out = run_model.run_model(basin, calib, huc4='9999')   # huc4 not a Great Lake
print('output columns:', list(out.columns))
out[['NHDPlusID','waterbody','CO2_ppm','k_co2_m_s','FCO2_gC_m2_yr']].round(3)

## 7. Sanity checks

Two checks that the port behaves physically:
1. With no benthic and no water-column source (`C_bz=0, F_wc=0`), every reach's CO₂ must lie
   between atmospheric (400) and groundwater (16000): nothing can create CO₂ out of nothing.
2. Raising `Cbz_riv` must raise the river median (an identifiable knob).


In [ ]:
h0 = calibrate.run_forward(basin, (0.0, 0.0, 0.0, 0.0))
co2 = h0['CO2_ppm'].to_numpy()
print('check 1: C_bz=0, F_wc=0 ->  CO2 in [%.0f, %.0f] ?  %s'
      % (co2.min(), co2.max(), bool((co2>=C_ATMOSPHERE_PPM-1).all() and (co2<=C_GROUNDWATER_PPM+1).all())))

lo = calibrate.evaluate(basin,(200,600,1e-4,1e-5),**{'riverCO2':targets['riverCO2'],'lakeCO2':targets['lakeCO2']}).river_median
hi = calibrate.evaluate(basin,(1800,600,1e-4,1e-5),**{'riverCO2':targets['riverCO2'],'lakeCO2':targets['lakeCO2']}).river_median
print('check 2: river median rises with Cbz_riv ?  %.1f -> %.1f  %s' % (lo, hi, hi>lo))

## 8. Matt's question, in code: which knobs are identifiable?

Matt suspected `F_wc` may not move the basin medians much, i.e. it may not be
*identifiable*. We can test it directly: sweep each knob across its full range (others fixed) and
see how far the median moves. Because rivers have short residence time and lakes have long
residence time, we expect `Fwc_riv` to be nearly flat and `Fwc_lake` to matter. This is a preview
of the real single-basin sweep.


In [ ]:
base = [600.0, 600.0, 5e-3, 5e-3]
tgt = dict(riverCO2=targets['riverCO2'], lakeCO2=targets['lakeCO2'])
grid = np.linspace(0.0, 1.0, 21)   # fraction of each knob's range

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5)); axes = axes.ravel()
for k, name in enumerate(PARAM_NAMES):
    lo, hi = LOWER_BOUNDS[k], UPPER_BOUNDS[k]
    river_meds, lake_meds = [], []
    for g in grid:
        p = list(base); p[k] = lo + g*(hi-lo)
        e = calibrate.evaluate(basin, tuple(p), **tgt)
        river_meds.append(e.river_median); lake_meds.append(e.lake_median)
    ax = axes[k]
    ax.plot(grid, river_meds, color='#2563eb', lw=2.5, label='river median')
    ax.plot(grid, lake_meds,  color='#d97706', lw=2.5, label='lake median')
    rng_r = max(river_meds)-min(river_meds); rng_l = max(lake_meds)-min(lake_meds)
    ax.set_title(f'{name}\nriver moves {rng_r:,.0f} | lake moves {rng_l:,.0f} ppm', fontsize=10)
    ax.set_xlabel('knob value (min -> max)'); ax.set_ylabel('basin median pCO2 [ppm]')
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=9)
fig.suptitle('Sensitivity of basin medians to each knob (synthetic basin)', fontweight='bold')
plt.tight_layout(); plt.show()
print('Cbz knobs move the medians a lot (identifiable). Fwc_riv is nearly flat (short river HRT);',
      'Fwc_lake matters (long lake HRT). This is exactly the pattern Matt asked about.')

## 9. What is faithful, what changed, what is next

EN.
- Faithful (flow unchanged): `CO2_MODEL` equation + all branches, the upstream->downstream
  reach loop, the median-per-water-body fitness, the physics helpers, the emission formula.
- Efficiency-only changes (identical numbers): numpy vectorization; an O(N) upstream-index
  map instead of the per-reach `which()` scan.
- Found + kept faithfully: a unit quirk in `kco2_func` (lake area passed in m² vs km² classes)
: replicated, not silently 'fixed', and documented in the code.
- Found + fixed (were real bugs in the first port): the cross-basin import table must be
  filtered to the current basin before appending; `HydroSeq` NaN rows must be dropped like R.
- Deferred: `setup_hydrography` needs the NHD-HR `.gdb` files (on NAS) + AHG `.rds`
  coefficients; it is a documented structural stub. Use `synthetic_basin` until data is wired in.
- Next: run the real single-basin parameter sweep (Section 8, but on a real basin) to
  confirm linearity and measure `Fwc` identifiability: the experiment Matt proposed.
